### House Rent Prediction using Linear Regression

In [5]:
# Importing Libraries
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [49]:
# Configurations
DATA_PATH = "House rent in Ibadan.csv"
TARGET_COLUMN = "annual_rent_ngn"

MODEL_PATH = "rent_model.pkl"
SCALER_PATH = "scaler.pkl"
FEATURE_PATH = "feature_order.pkl"

print("Configuration loaded.")


Configuration loaded.


In [50]:
# Loading the data
df = pd.read_csv(DATA_PATH)
print("Dataset Loaded Successfully")
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())


Dataset Loaded Successfully
Dataset shape: (10000, 11)
Columns: ['Location', 'Property_Type', 'Bedrooms', 'Bathrooms', 'House_Size_sqm', 'Lot_Size_sqm', 'Year_Built', 'Furnishing', 'Water_Supply', 'Power_Supply', 'Annual_Rent_NGN']


In [29]:
# Check data info
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Location         10000 non-null  object
 1   Property_Type    10000 non-null  object
 2   Bedrooms         10000 non-null  int64 
 3   Bathrooms        10000 non-null  int64 
 4   House_Size_sqm   10000 non-null  int64 
 5   Lot_Size_sqm     10000 non-null  int64 
 6   Year_Built       10000 non-null  int64 
 7   Furnishing       10000 non-null  object
 8   Water_Supply     10000 non-null  object
 9   Power_Supply     10000 non-null  object
 10  Annual_Rent_NGN  10000 non-null  int64 
dtypes: int64(6), object(5)
memory usage: 859.5+ KB
None


In [30]:
# Check for null values
print(df.isnull().sum())

Location           0
Property_Type      0
Bedrooms           0
Bathrooms          0
House_Size_sqm     0
Lot_Size_sqm       0
Year_Built         0
Furnishing         0
Water_Supply       0
Power_Supply       0
Annual_Rent_NGN    0
dtype: int64


In [32]:
# Remove duplicates and missing values
df = df.drop_duplicates()
df.dropna(inplace=True)
print("Cleaned Dataset Shape:", df.shape)

Cleaned Dataset Shape: (10000, 11)


In [33]:
# Standardise column names to lowercase with underscores
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
print("Columns:", df.columns.tolist())

Columns: ['location', 'property_type', 'bedrooms', 'bathrooms', 'house_size_sqm', 'lot_size_sqm', 'year_built', 'furnishing', 'water_supply', 'power_supply', 'annual_rent_ngn']


In [34]:
# Show unique values for categorical columns (so app.py dropdowns match)
for col in ['location', 'property_type', 'furnishing', 'water_supply', 'power_supply']:
    print(f"{col}: {sorted(df[col].unique().tolist())}")

location: ['Agbowo', 'Ajibode', 'Akobo', 'Apata', 'Bodija', 'Challenge', 'Mokola', 'Ologuneru', 'Oluyole', 'Ring Road']
property_type: ['2 Bedroom Flat', '3 Bedroom Flat', 'Duplex', 'Mini Flat', 'Room', 'Self Contained']
furnishing: ['Fully Furnished', 'Semi-Furnished', 'Unfurnished']
water_supply: ['Borehole', 'Public', 'Well']
power_supply: ['Moderate', 'Poor', 'Stable']


In [35]:
# Feature and Target Separation
X = df.drop(TARGET_COLUMN, axis=1)
y = df[TARGET_COLUMN]
print("Features:", X.columns.tolist())
print("Target:", TARGET_COLUMN)

Features: ['location', 'property_type', 'bedrooms', 'bathrooms', 'house_size_sqm', 'lot_size_sqm', 'year_built', 'furnishing', 'water_supply', 'power_supply']
Target: annual_rent_ngn


In [36]:
# ONE-HOT ENCODE categorical variables (drop_first=True to avoid multicollinearity)
# NOTE: We use get_dummies only — no LabelEncoder needed
X_encoded = pd.get_dummies(X, drop_first=True)
print("Encoded feature count:", X_encoded.shape[1])
print("Encoded columns:", X_encoded.columns.tolist())

# Save the feature order AFTER encoding — this is what the model is trained on
FEATURE_ORDER = X_encoded.columns.tolist()

Encoded feature count: 25
Encoded columns: ['bedrooms', 'bathrooms', 'house_size_sqm', 'lot_size_sqm', 'year_built', 'location_Ajibode', 'location_Akobo', 'location_Apata', 'location_Bodija', 'location_Challenge', 'location_Mokola', 'location_Ologuneru', 'location_Oluyole', 'location_Ring Road', 'property_type_3 Bedroom Flat', 'property_type_Duplex', 'property_type_Mini Flat', 'property_type_Room', 'property_type_Self Contained', 'furnishing_Semi-Furnished', 'furnishing_Unfurnished', 'water_supply_Public', 'water_supply_Well', 'power_supply_Poor', 'power_supply_Stable']


In [37]:
# Split dataset
print("\nSplitting Dataset....")

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    random_state=42
)

print("Training Size:", X_train.shape)
print("Testing Size:", X_test.shape)


Splitting Dataset....
Training Size: (8000, 25)
Testing Size: (2000, 25)


In [39]:
# Feature Scaling — fit ONLY on training data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Scaling Completed.")

Scaling Completed.


In [40]:
# Train model
model = LinearRegression()
model.fit(X_train_scaled, y_train)
print("Model Training Complete.")

Model Training Complete.


In [41]:
y_pred = model.predict(X_test_scaled)

In [42]:
# Model Evaluation
MAE = mean_absolute_error(y_test, y_pred)
MSE = mean_squared_error(y_test, y_pred)
RMSE = np.sqrt(MSE)
R2 = r2_score(y_test, y_pred)

print("MAE:", MAE)
print("MSE:", MSE)
print("RMSE:", RMSE)
print("R2 Score:", R2)

MAE: 71949.19273838389
MSE: 13016784372.036915
RMSE: 114091.1231079654
R2 Score: 0.9280392385884184


In [43]:
# Cross Validation
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X_encoded, y, cv=kfold, scoring="r2")

print("Cross Validation Scores:", scores)
print("Average CV Score:", scores.mean())

Cross Validation Scores: [0.92803924 0.9105534  0.92472002 0.92222235 0.92891054]
Average CV Score: 0.9228891099376906


In [44]:
# Unit testing — sample from encoded data
sample = X_encoded.iloc[[0]]
sample_scaled = scaler.transform(sample)
prediction = model.predict(sample_scaled)
assert prediction[0] > 0, "Prediction Test Failed"
print("Unit Test Passed. Sample prediction: ₦{:,.0f}".format(prediction[0]))

Unit Test Passed. Sample prediction: ₦160,156


In [45]:
# Save model artifacts
joblib.dump(model, MODEL_PATH)
joblib.dump(scaler, SCALER_PATH)
joblib.dump(FEATURE_ORDER, FEATURE_PATH)

print("Model Saved:")
print(MODEL_PATH)
print(SCALER_PATH)
print(FEATURE_PATH)
print("\nFeature order saved:", FEATURE_ORDER)


Model Saved:
rent_model.pkl
scaler.pkl
feature_order.pkl

Feature order saved: ['bedrooms', 'bathrooms', 'house_size_sqm', 'lot_size_sqm', 'year_built', 'location_Ajibode', 'location_Akobo', 'location_Apata', 'location_Bodija', 'location_Challenge', 'location_Mokola', 'location_Ologuneru', 'location_Oluyole', 'location_Ring Road', 'property_type_3 Bedroom Flat', 'property_type_Duplex', 'property_type_Mini Flat', 'property_type_Room', 'property_type_Self Contained', 'furnishing_Semi-Furnished', 'furnishing_Unfurnished', 'water_supply_Public', 'water_supply_Well', 'power_supply_Poor', 'power_supply_Stable']


In [46]:
# Verify saved model
loaded_model = joblib.load(MODEL_PATH)
test_prediction = loaded_model.predict(X_test_scaled[:1])
print("Loaded Model Prediction: ₦{:,.0f}".format(test_prediction[0]))

Loaded Model Prediction: ₦777,260


In [47]:
print("\nHOUSE RENT PREDICTION PIPELINE COMPLETED SUCCESSFULLY")


HOUSE RENT PREDICTION PIPELINE COMPLETED SUCCESSFULLY
